<a href="https://colab.research.google.com/github/srihanreddy/Reinforcement-Learning/blob/main/practical_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Creating the Environment Module
We will first write the grid world module helper `grid_env_det.py` to represent states, actions, transitions, and rewards as described in the requirements.

In [1]:
%%writefile grid_env_det.py
# grid_env_det.py

class GridWorldDet:
    def __init__(self):
        # Layout:
        # S0(0,0)  S1(0,1)  G(0,2)
        # X(1,0)   S3(1,1)  S4(1,2)
        self.states = {
            (0, 0): "S0",
            (0, 1): "S1",
            (0, 2): "G",
            (1, 1): "S3",
            (1, 2): "S4"
        }
        self.obstacle = (1, 0)
        self.goal = (0, 2)
        self.actions = ["Up", "Down", "Left", "Right"]

    def step(self, state_name, action):
        # Map name to coords
        coords_map = {v: k for k, v in self.states.items()}
        curr_coords = coords_map[state_name]
        r, c = curr_coords

        # Define movement offsets
        if action == "Up":
            next_coords = (r - 1, c)
        elif action == "Down":
            next_coords = (r + 1, c)
        elif action == "Left":
            next_coords = (r, c - 1)
        elif action == "Right":
            next_coords = (r, c + 1)
        else:
            raise ValueError(f"Invalid Action: {action}")

        # 1. Attempting to enter the obstacle (X)
        if next_coords == self.obstacle:
            return None, -5, True # Terminal, state is dead

        # 2. Out of bounds check
        if not (0 <= next_coords[0] <= 1 and 0 <= next_coords[1] <= 2):
            # remains in the same state, normal movement reward
            return state_name, -1, False

        # 3. Successful transitions
        next_state_name = self.states[next_coords]

        if next_state_name == "G":
            return "G", 10, True # Reaching goal terminates episode

        # Normal movement
        return next_state_name, -1, False

Writing grid_env_det.py


### Implementing Assignment 1 Functions and Demonstration

In [2]:
import numpy as np
from grid_env_det import GridWorldDet

# Instantiate the environment
env = GridWorldDet()

# 4. Policy Implementation
policy = {
    "S0": "Right",
    "S1": "Down",
    "S3": "Right",
    "S4": "Up"
}

# 5. Required Functions

def state_value_function(state, verbose=True):
    """
    Calculates state value V(s) by following the predefined policy.
    Returns the discounted return and the sequence steps.
    """
    current_state = state
    gamma = 0.9
    rewards = []
    steps = []
    episode_ended = False

    if verbose:
        print(f"\n--- Calculating V({state}) ---")
        print(f"Start State: {current_state}")

    while not episode_ended:
        if current_state not in policy:
            # Terminal or invalid state
            break
        action = policy[current_state]
        next_state, reward, done = env.step(current_state, action)

        rewards.append(reward)
        steps.append((current_state, action, reward, next_state if next_state else 'Obstacle (X)'))

        if verbose:
            print(f"Step {len(steps)}: From {current_state} take action '{action}' -> Reward: {reward}, Next State: {next_state if next_state else 'X (Terminal)'}")

        current_state = next_state
        if done:
            episode_ended = True

    # Calculate discounted return
    discounted_return = sum((gamma**i) * r for i, r in enumerate(rewards))

    if verbose:
        # Format formula representation
        formula_terms = [f"({gamma}^{i}) * ({r})" if i > 0 else f"{r}" for i, r in enumerate(rewards)]
        print(f"Rewards sequence: {rewards}")
        print(f"Formula: {' + '.join(formula_terms)}")
        print(f"V({state}) = {discounted_return:.4f}")

    return discounted_return


def action_value_function(state, action, verbose=True):
    """
    Calculates action value Q(s, a) by executing the specified action first,
    then following the policy for subsequent transitions.
    """
    gamma = 0.9
    rewards = []
    steps = []

    if verbose:
        print(f"\n--- Calculating Q({state}, {action}) ---")
        print(f"Start State: {state}, Executing initial action: {action}")

    # First Step
    next_state, reward, done = env.step(state, action)
    rewards.append(reward)
    steps.append((state, action, reward, next_state if next_state else 'Obstacle (X)'))

    if verbose:
        print(f"Step 1: From {state} take Action '{action}' -> Reward: {reward}, Next State: {next_state if next_state else 'X (Terminal)'}")

    # Subsequent steps
    current_state = next_state
    while not done:
        if current_state not in policy:
            break
        next_action = policy[current_state]
        next_state, reward, done = env.step(current_state, next_action)

        rewards.append(reward)
        steps.append((current_state, next_action, reward, next_state if next_state else 'Obstacle (X)'))

        if verbose:
            print(f"Step {len(steps)}: From {current_state} take action '{next_action}' -> Reward: {reward}, Next State: {next_state if next_state else 'X (Terminal)'}")

        current_state = next_state

    # Calculate discounted return
    discounted_return = sum((gamma**i) * r for i, r in enumerate(rewards))

    if verbose:
        formula_terms = [f"({gamma}^{i}) * ({r})" if i > 0 else f"{r}" for i, r in enumerate(rewards)]
        print(f"Rewards sequence: {rewards}")
        print(f"Formula: {' + '.join(formula_terms)}")
        print(f"Q({state}, {action}) = {discounted_return:.4f}")

    return discounted_return

# --- Display Output Demonstration ---
print("=============================================================")
print("1. Grid World Environment Design")
print("   Layout:")
print("   | S0 (0,0) | S1 (0,1) | G (0,2)  |")
print("   | X (1,0)  | S3 (1,1) | S4 (1,2) |")
print("\n2. Environment Specifications:")
print("   - Normal movement reward: -1")
print("   - Goal reward: +10")
print("   - Obstacle reward: -5")
print("   - Discount factor (gamma): 0.9")
print("\n3. Defined Policy:")
for k, v in policy.items():
    print(f"   State {k} -> Action: {v}")
print("=============================================================")

# Execution of Tasks
v_s0 = state_value_function("S0")
q_s0_up = action_value_function("S0", "Up")

1. Grid World Environment Design
   Layout:
   | S0 (0,0) | S1 (0,1) | G (0,2)  |
   | X (1,0)  | S3 (1,1) | S4 (1,2) |

2. Environment Specifications:
   - Normal movement reward: -1
   - Goal reward: +10
   - Obstacle reward: -5
   - Discount factor (gamma): 0.9

3. Defined Policy:
   State S0 -> Action: Right
   State S1 -> Action: Down
   State S3 -> Action: Right
   State S4 -> Action: Up

--- Calculating V(S0) ---
Start State: S0
Step 1: From S0 take action 'Right' -> Reward: -1, Next State: S1
Step 2: From S1 take action 'Down' -> Reward: -1, Next State: S3
Step 3: From S3 take action 'Right' -> Reward: -1, Next State: S4
Step 4: From S4 take action 'Up' -> Reward: 10, Next State: G
Rewards sequence: [-1, -1, -1, 10]
Formula: -1 + (0.9^1) * (-1) + (0.9^2) * (-1) + (0.9^3) * (10)
V(S0) = 4.5800

--- Calculating Q(S0, Up) ---
Start State: S0, Executing initial action: Up
Step 1: From S0 take Action 'Up' -> Reward: -1, Next State: S0
Step 2: From S0 take action 'Right' -> Reward: -